<a href="https://colab.research.google.com/github/Adhiaris/Grokking-Deep-Learning/blob/main/chapter_11_word_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 11: Neural Networks That Understand Language
### Word Embeddings and king – man + woman ≈ queen

This chapter introduces natural language processing (NLP) using neural networks. The central idea is that words can be represented as dense vectors (embeddings) that encode semantic meaning.

## 1. From Words to Numbers: One-Hot Encoding

Before feeding text into a network, words must be converted to numbers. The simplest method is one-hot encoding: a vector of zeros with a single 1 at the word's index.

In [ ]:
import numpy as np

vocab = ["the", "cat", "sat", "on", "mat", "dog", "ran", "away"]
word2idx = {w: i for i, w in enumerate(vocab)}

def one_hot(word, vocab_size):
    vec = np.zeros(vocab_size)
    vec[word2idx[word]] = 1
    return vec

for word in ["cat", "dog", "mat"]:
    vec = one_hot(word, len(vocab))
    print(f"{word:<8}: {vec.astype(int)}")

print(f"\nVocab size: {len(vocab)} — each word is a {len(vocab)}-dimensional sparse vector.")

## 2. The Embedding Layer

An embedding layer is a lookup table: a matrix W of shape (vocab_size, embed_dim). The embedding for word i is simply row i of W. This is equivalent to multiplying the one-hot vector by W, but far more efficient.

In [ ]:
import numpy as np

np.random.seed(42)

vocab     = ["the", "cat", "sat", "on", "mat", "dog", "ran", "away"]
word2idx  = {w: i for i, w in enumerate(vocab)}
vocab_size = len(vocab)
embed_dim  = 3

W_embed = np.random.rand(vocab_size, embed_dim) - 0.5

print("Embedding matrix shape:", W_embed.shape)
print()

for word in ["cat", "dog"]:
    idx = word2idx[word]
    emb = W_embed[idx]
    print(f"Embedding for '{word}' (row {idx}): {np.round(emb, 3)}")

## 3. Training a Sentiment Classifier

A simple NLP task: given a movie review (bag of words), predict positive or negative sentiment. The network learns embeddings that capture whether words are positive or negative.

In [ ]:
import numpy as np

np.random.seed(0)

vocab    = ["good", "bad", "great", "terrible", "love", "hate", "movie", "film"]
word2idx = {w: i for i, w in enumerate(vocab)}

reviews = [
    (["good", "great", "love", "movie"], 1),
    (["bad", "terrible", "hate", "film"], 0),
    (["great", "love", "film"], 1),
    (["bad", "hate", "movie"], 0),
    (["good", "movie"], 1),
    (["terrible", "film"], 0),
]

def to_bag_of_words(words):
    vec = np.zeros(len(vocab))
    for w in words:
        if w in word2idx:
            vec[word2idx[w]] = 1
    return vec

W = 2*np.random.rand(len(vocab), 1) - 1
alpha = 0.1

def sigmoid(x): return 1 / (1 + np.exp(-x))

print(f"{'Epoch':<8} {'Error':>12}")
print("-" * 22)

for epoch in range(50):
    total_err = 0
    for words, label in reviews:
        x    = to_bag_of_words(words)
        pred = sigmoid(x.dot(W))[0]
        err  = (pred - label) ** 2
        total_err += err
        delta = pred - label
        W -= alpha * x.reshape(-1, 1) * delta * pred * (1 - pred)
    if epoch % 10 == 0:
        print(f"{epoch:<8} {total_err:>12.6f}")

## 4. Word Similarity via Embeddings

After training, semantically similar words should have similar embedding vectors. Cosine similarity measures the angle between two vectors — a score of 1 means identical direction.

In [ ]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

np.random.seed(7)
embed_dim  = 5
vocab      = ["king", "queen", "man", "woman", "cat", "dog"]

embeddings = {
    "king"  : np.array([0.9,  0.8, 0.2, 0.1, 0.05]),
    "queen" : np.array([0.85, 0.8, 0.1, 0.9, 0.05]),
    "man"   : np.array([0.8,  0.2, 0.9, 0.1, 0.05]),
    "woman" : np.array([0.75, 0.2, 0.1, 0.9, 0.05]),
    "cat"   : np.array([0.05, 0.1, 0.2, 0.2, 0.95]),
    "dog"   : np.array([0.05, 0.1, 0.2, 0.2, 0.90]),
}

pairs = [("king", "queen"), ("man", "woman"), ("king", "cat"), ("cat", "dog")]

print(f"{'Pair':<20} {'Cosine Similarity':>18}")
print("-" * 42)
for a, b in pairs:
    sim = cosine_similarity(embeddings[a], embeddings[b])
    print(f"{a+' & '+b:<20} {sim:>18.4f}")

## 5. Word Analogy: king – man + woman ≈ queen

A famous property of well-trained embeddings: vector arithmetic encodes semantic relationships.

In [ ]:
import numpy as np

embeddings = {
    "king"  : np.array([0.9,  0.8, 0.2, 0.1, 0.05]),
    "queen" : np.array([0.85, 0.8, 0.1, 0.9, 0.05]),
    "man"   : np.array([0.8,  0.2, 0.9, 0.1, 0.05]),
    "woman" : np.array([0.75, 0.2, 0.1, 0.9, 0.05]),
    "cat"   : np.array([0.05, 0.1, 0.2, 0.2, 0.95]),
    "dog"   : np.array([0.05, 0.1, 0.2, 0.2, 0.90]),
}

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

analogy_vec = embeddings["king"] - embeddings["man"] + embeddings["woman"]

scores = {}
for word, vec in embeddings.items():
    if word not in ["king", "man", "woman"]:
        scores[word] = cosine_similarity(analogy_vec, vec)

best = max(scores, key=scores.get)

print("king - man + woman = ?")
print()
for word, score in sorted(scores.items(), key=lambda x: -x[1]):
    print(f"  {word:<10} similarity: {score:.4f}")
print(f"\nBest answer: {best}")